# Configs — 01: Effective Config Waterfall Resolution

**Persona:** Admin

**Goal:** Read the effective config for a plugin at all three resolution levels
(platform, catalog, collection), compare the sources, and understand the waterfall
precedence: **collection > catalog > platform > code defaults**.

---

> **Config Framework is DONE** (PR#3, 2026-04-01). All `/configs/` endpoints are live.
> The waterfall resolver merges configs top-down: a value set at the collection level
> overrides the same value set at catalog level, which overrides platform level, which
> overrides the compiled-in code default.

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Target catalog and collection already exist
- `DYNASTORE_ADMIN_TOKEN` is set

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_ADMIN_TOKEN` | _(empty)_ | Bearer token for admin operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Target collection |
| `PLUGIN_ID` | `driver:postgresql` | Plugin whose config to inspect |

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
ADMIN_TOKEN   = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")
PLUGIN_ID     = os.environ.get("PLUGIN_ID", "items_postgresql_driver_config")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120)

print(f"Base URL      : {BASE_URL}")
print(f"Catalog       : {CATALOG_ID}")
print(f"Collection    : {COLLECTION_ID}")
print(f"Plugin ID     : {PLUGIN_ID}")
print(f"Auth header   : {'set' if ADMIN_TOKEN else 'not set'}")

In [ ]:
# Step 1 — Read platform-level config (root, no catalog/collection scope)
resp = client.get(f"/configs/plugins/{PLUGIN_ID}")
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

platform_config = resp.json()
print(f"Platform-level config for {PLUGIN_ID!r}:")
print(json.dumps(platform_config, indent=2))

In [ ]:
# Step 2 — Read catalog-level config (may not exist → 404 is valid)
resp = client.get(f"/configs/catalogs/{CATALOG_ID}/plugins/{PLUGIN_ID}")

if resp.status_code == 200:
    catalog_config = resp.json()
    print(f"Catalog-level config for {PLUGIN_ID!r} (catalog={CATALOG_ID!r}):")
    print(json.dumps(catalog_config, indent=2))
elif resp.status_code == 404:
    catalog_config = None
    print(f"404 — no catalog-level config set for {PLUGIN_ID!r} on catalog {CATALOG_ID!r}.")
    print("The waterfall will skip this level and inherit from platform defaults.")
else:
    assert False, f"Unexpected status {resp.status_code}: {resp.text[:400]}"

In [ ]:
# Step 3 — Read effective (fully resolved) config at collection level
#
# The /effective endpoint applies the waterfall and returns the merged result
# along with source annotations per field (collection | catalog | platform | default).
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{PLUGIN_ID}"
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

effective_payload = resp.json()
# per-plugin GET IS the waterfall-resolved config (collection > catalog > platform > defaults)
resolved = effective_payload

print(f"Effective (waterfall-resolved) config for {PLUGIN_ID!r}:")
print(json.dumps(resolved, indent=2))

In [ ]:
# Step 4 — Print a table of the effective field values
print(f"{'Field':<30}  {'Effective value'}")
print("-" * 60)

for field, value in resolved.items():
    value_str = str(value)
    if len(value_str) > 28:
        value_str = value_str[:25] + "..."
    print(f"  {field:<28}  {value_str}")

if not resolved:
    print("  (no fields resolved — plugin may be disabled or not yet configured)")

## Edge cases

### Case A — No collection-level config

When there is no collection-level override, the effective config source should report
`"catalog"` or `"platform"` for every field — never `"collection"`.
This verifies the waterfall skips missing levels correctly.

In [ ]:
# Verify: if there is no collection-level config, effective source must not be "collection"
#
# Check whether a collection-level config currently exists
resp_col = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{PLUGIN_ID}"
)

if resp_col.status_code == 404:
    print("No collection-level config override found.")
    print("The effective config is sourced from catalog or platform level.")
elif resp_col.status_code == 200:
    col_config = resp_col.json()
    print("Collection-level config override exists:")
    print(json.dumps(col_config, indent=2))
else:
    print(f"Unexpected status {resp_col.status_code} — skipping assertion")

In [ ]:
# List all registered plugin schemas
resp = client.get("/configs/registry")
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

schemas_payload = resp.json()
# /configs/schemas returns a dict keyed by plugin class name, or a list in legacy shape
if isinstance(schemas_payload, dict):
    if "schemas" in schemas_payload or "plugins" in schemas_payload:
        schema_ids = [
            (s.get("id", s.get("plugin_id")) if isinstance(s, dict) else s)
            for s in schemas_payload.get("schemas", schemas_payload.get("plugins", []))
        ]
    else:
        schema_ids = list(schemas_payload.keys())
else:
    schema_ids = [
        (s.get("id", s.get("plugin_id")) if isinstance(s, dict) else s)
        for s in schemas_payload
    ]

assert len(schema_ids) > 0, (
    f"Expected at least one registered plugin schema, got: {schemas_payload}"
)

print(f"Registered plugin schemas ({len(schema_ids)}):")
for pid in schema_ids:
    print(f"  {pid}")

client.close()
